# add missing exchange and transport reactions to models

In [119]:
import sys
sys.path.insert(0, '/home/emma/Dokumente/thesis')

from collections import defaultdict
import os
from cobra.io import read_sbml_model, write_sbml_model
from functions import *
import pandas as pd
import copy

In [120]:
#model_dir = "/home/emma/Dokumente/thesis/Model_generation_curation/Draft_models/Models"
model_dir = "/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/mb1_mdr_rdr_dp_lib_bz"
model_dir_save = "/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/add_exr"

In [121]:
seqalign_glucimport = pd.read_csv("/home/emma/Dokumente/thesis/Model_generation_curation/blast/blast_results/glucose/highest_seqid_per_organism.csv")

In [122]:
seq2class_gluc = {"ABC": ["MGLB_ECOLI"], "PTS": ["A0A423H885_9PSED", "PTGCB_ECOLI", "PTOCB_ECOLI", "A0A1B1ZA10_9BACL"], "Proton_symport": ["A0A167YAG0_9FLAO","GLCP_STAES", "GALP_ECOL6", "T0GBX4_9SPHN", "V6S0T5_9FLAO"], "SWEET" : ["SWEET_LEPBP"]}

#### load models and medium, check exchange reactions

In [123]:
loaded_model_dict = {}
for file in os.listdir(model_dir_save):
    if not file.endswith(('.xml', '.sbml')):
        continue
        
    model = read_sbml_model(os.path.join(model_dir_save, file))
    model_id = model.id
    loaded_model_dict[model_id] = model

In [124]:
TSB = pd.read_csv("/home/emma/Dokumente/thesis/media_creation/TSB_medium.csv", header=None)
TSB_highbounds = TSB.copy()
TSB_highbounds[1] = 1000
TSB_dict = dict(zip(TSB_highbounds[0], TSB_highbounds[1]))

In [125]:
af7 = pd.read_csv("/home/emma/Dokumente/thesis/media_creation/created_media/combined_af.csv", header=None)
af7_dict = dict(zip(af7[0], af7[1]))

In [126]:
no_exr = defaultdict(list)

mapped_metabolites = {}
for ex_r in af7_dict.keys():
    base = ex_r[3:] if ex_r.startswith("EX_") else ex_r
    
    if base.endswith("_m"): base = base[:-2]
    elif base.endswith("_medium"): base = base[:-7]
    elif base.endswith("_e"): base = base[:-2]
        
    mapped_metabolites[ex_r] = {
        "cytosol": base + "_c",
        "strain_exchange": "EX_" + base + "_e"  # Map to the specific strain exchange format
    }

for model_id, model in loaded_model_dict.items():
    model_reactions = set(model.reactions.list_attr("id"))
    model_metabolites = set(model.metabolites.list_attr("id"))
    
    for ex_r, targets in mapped_metabolites.items():
        met_c = targets["cytosol"]
        strain_ex = targets["strain_exchange"]
        
        if met_c in model_metabolites and strain_ex not in model_reactions:
            no_exr[ex_r].append(model_id)

df_missing_exchanges = pd.DataFrame.from_dict(no_exr, orient='index')

In [127]:
df_missing_exchanges.to_csv("test_miss_ex.csv")

In [128]:
df_missing_exchanges = pd.DataFrame.from_dict(no_exr, orient='index')

In [129]:
count = 0
for modid, mod in loaded_model_dict.items():
    if "succ_c" in mod.metabolites:
        print(modid)
        count += 1
print(count)

m_504
m_364
m_778
m_1234
m_1114
m_428
m_892
m_939
m_790
m_895
m_1432
m_163
m_1056
m_1101
m_1208
m_946
m_2862
m_397
m_459
m_262
m_638
m_100
m_997
m_1334
m_2751
m_2774
m_709
m_868
m_1357
m_761
m_1080
m_1350
m_947
m_1362
m_793
m_1167
m_644
m_978
m_867
39


## Functions

In [130]:
def add_amino_acids(model, ex_r):
    """
    Checks if a metabolite's cytosolic form exists, and if so,
    executes the mapped functions to add extracellular components and pathways.
    """
    met_clean = ex_r.replace("EX_", "")
    if met_clean.endswith("_e"):
        met_clean = met_clean[:-2]
    elif met_clean.endswith("_m"):
        met_clean = met_clean[:-2]
        
    met_c = met_clean + "_c"
    # Check if the cytosolic version exists in the COBRA model
    if met_c in model.metabolites:
        if met_clean in AMINO_ACID_MAP:
            AMINO_ACID_MAP[met_clean]["mets"](model)
            AMINO_ACID_MAP[met_clean]["rxns"](model)
            print(f"Successfully integrated pathways for {met_clean}")
        else:
            print(f"Warning: Met '{met_clean}' found in model, but no function mapping exists.")
    else:
        print(f"Skipped: Cytosolic metabolite '{met_c}' is not present in the model.")
    if "ASPte" in model.reactions:
        model.remove_reactions(["ASPte"])
        add_new_rxn(model, "ASPabc", "L-Aspartate transport via ABC system", 0.0, 1000.0, {"asp__L_e": -1.0, "atp_c": -1.0, "h2o_c": -1.0, "adp_c": 1.0, "pi_c": 1.0, "h_c": 1.0, "asp__L_c": 1.0})
        

In [131]:
def add_non_aa(model_id, model):
    if model_id in no_exr["EX_na1_e"]:
        add_na1_mets(model)
        add_na1_rxn(model)
    if model_id in no_exr["EX_fe2_e"]:
        add_fe2_mets(model)
        add_fe2_rxn(model)
    if model_id in no_exr["EX_pi_e"]:
        add_pi_mets(model)
        add_pi_rxn(model)
    if model_id in no_exr["EX_so4_e"]:
        add_so4_mets(model)
        add_so4_rxn(model)

In [132]:
def get_gluc_importer(model, seqalign_glucimport, seq2class_gluc):

    model_id = int(model.id.split("_")[1])
    matched_rows = seqalign_glucimport[seqalign_glucimport["organism"] == model_id]
    
    if matched_rows.empty:
        print(f"Warning: No sequence alignment data found for organism {model_id}")
        return None
        
    trans_rxn = matched_rows["subject_id"].iloc[0]
    gluc_class = None
    for key, values_list in seq2class_gluc.items():
        if trans_rxn in values_list:
            gluc_class = key
        
    return gluc_class

### add reaction by reaction

#### EX_fru_e

In [133]:
def add_fru_mets(model):
    add_new_met(model, "fru_e", "D-Fructose", "C6H12O6", 0, "e")

def add_fru_rxn(model):
    add_new_rxn(model, "EX_fru_e", "D-Fructose exchange", -1000.0, 1000.0, {"fru_e": -1.0})
    add_new_rxn(model, "FRUpts", "D-Fructose transport via PEP:Pyr PTS", -1000.0, 1000.0, {"fru_e": -1.0, "pep_c": -1.0, "f6p_c": 1.0, "pyr_c":1.0})



#### EX_glc__D_e

In [134]:
def add_glc_all(model):
    add_new_met(model, "glc__D_e", "D-Glucose", "C6H12O6", 0, "e")
    add_new_met(model, "g6p_c", "D-Glucose 6-phosphate", "C6H11O9P", -2, "c")
    add_new_met(model, "f6p_c", "D-Fructose 6-phosphate", "C6H11O9P", -2, "c")
    add_new_met(model, "fdp_c", "D-Fructose 1,6-bisphosphate", "C6H10O12P2", -4, "c")
    add_new_met(model, "dhap_c", "Dihydroxyacetone phosphate", "C3H5O6P", -2, "c")
    add_new_met(model, "g3p_c", "Glyceraldehyde 3-phosphate", "C3H5O6P", -2, "c")
    add_new_met(model, "13dpg_c", "3-Phospho-D-glyceroyl phosphate", "C3H4O10P2", -4, "c")
    add_new_met(model, "3pg_c", "3-Phospho-D-glycerate", "C3H4O7P", -3, "c")
    add_new_met(model, "2pg_c", "2-Phospho-D-glycerate", "C3H4O7P", -3, "c")
    add_new_met(model, "pep_c", "Phosphoenolpyruvate", "C3H2O6P", -3, "c")

    add_new_rxn(model, "EX_glc__D_e", "D-Glucose exchange", -1000.0, 1000.0, {"glc__D_e": -1.0})
    
    cl = get_gluc_importer(model, seqalign_glucimport, seq2class_gluc) #check which glucose transport is valid
    
    if cl == "ABC":
        add_new_rxn(model, "GLCabc", "D-glucose transport via ABC system", -1000.0, 1000.0, {"glc__D_e": -1.0, "h20_c": -1.0, "atp_c": -1.0, "adp_c": 1.0, "glc__D_c": 1.0, "h_c":1.0, "pi_c": 1.0})
        add_new_rxn(model, "HEX1", "Hexokinase", -1000.0, 1000.0, {"glc__D_c": -1.0, "atp_c": -1.0, "g6p_c": 1.0, "h_c": 1.0, "adp_c": 1.0})

    if cl == "PTS":
        add_new_rxn(model, "GLCpts", "D-glucose transport via PEP:Pyr PTS", -1000.0, 1000.0, {"glc__D_e": -1.0, "pep_c": -1.0, "g6p_c": 1.0, "pyr_c":1.0})
    if cl == "Proton_symport":
        add_new_met(model, "glc__D_c", "D-Glucose", "C6H12O6", 0, "c")
        add_new_rxn(model, "GLCt2", "D-glucose transport in via proton symport", -1000.0, 1000.0, {"glc__D_e": -1.0, "h_e": -1.0, "glc__D_c": 1.0, "h_c": 1.0})
        add_new_rxn(model, "HEX1", "Hexokinase", -1000.0, 1000.0, {"glc__D_c": -1.0, "atp_c": -1.0, "g6p_c": 1.0, "h_c": 1.0, "adp_c": 1.0})

    if cl == "SWEET":
        add_new_met(model, "glc__D_c", "D-Glucose", "C6H12O6", 0, "c")
        add_new_rxn(model, "SWEET", "D-glucose transport in via SWEET transporters", -1000.0, 1000.0, {"glc__D_e": -1.0, "glc__D_c": 1.0})
        add_new_rxn(model, "HEX1", "Hexokinase", -1000.0, 1000.0, {"glc__D_c": -1.0, "atp_c": -1.0, "g6p_c": 1.0, "h_c": 1.0, "adp_c": 1.0})

    add_new_rxn(model, "PGI", "Glucose-6-phosphate isomerase", -1000.0, 1000.0, {"g6p_c": -1.0, "f6p_c": 1.0})
    add_new_rxn(model, "PFK", "Phosphofructokinase", -1000.0, 1000.0, {"f6p_c": -1.0, "atp_c": -1.0, "fdp_c": 1.0, "h_c": 1.0, "adp_c": 1.0})
    add_new_rxn(model, "FBA", "Fructose-bisphosphate aldolase", -1000, 1000, {"fdp_c": -1.0, "dhap_c": 1.0, "g3p_c": 1.0})
    add_new_rxn(model, "TPI", "Triose-phosphate isomerase", -1000, 1000, {"dhap_c": -1.0, "g3p_c": 1.0})    
    add_new_rxn(model, "GAPD", "Glyceraldehyde-3-phosphate dehydrogenase", -1000, 1000, {"nad_c": -1.0, "pi_c": -1.0, "g3p_c": -1.0, "13dpg_c": 1.0, "h_c": 1.0, "nadh_c": 1.0})
    add_new_rxn(model, "PGK", "Phosphoglycerate kinase", -1000.0, 1000.0, {"13dpg_c": -1.0, "atp_c": 1.0, "3pg_c": 1.0, "adp_c": -1.0})
    add_new_rxn(model, "PGM", "Phosphoglycerate mutase", -1000.0, 1000.0, {"3pg_c": -1.0, "2pg_c": 1.0})
    add_new_rxn(model, "ENO", "Enolase", -1000, 1000, {"2pg_c": -1.0, "h2o_c": 1.0, "pep_c": 1.0})
    add_new_rxn(model, "PYK", "Pyruvate kinase", -1000.0, 1000.0, {"pep_c": -1.0, "atp_c": 1.0, "pyr_c": 1.0, "h_c": -1.0, "adp_c": -1.0})

#### EX_pi_e

In [135]:
def add_pi_mets(model):
    add_new_met(model, "pi_e", "Phosphate", "HO4P", -2, "e")

In [136]:
def add_pi_rxn(model):
    add_new_rxn(model, "EX_pi_e", "Phosphate exchange", -1000.0, 1000.0, {"pi_e": -1.0})
    add_new_rxn(model,"PIt7", "Phosphate transport", 0.0, 1000.0, {"pi_e": -1.0, "na1_e": -3.0, "na1_c":3.0, "pi_c": 1.0})


### EX_so4_e

In [137]:
def add_so4_mets(model):
    add_new_met(model, "so4_e", "Sulfate", "O4S", -2, "e")
def add_so4_rxn(model):
    add_new_rxn(model, "EX_so4_e", "Sulfate exchange", -1000.0, 1000.0, {"so4_e": -1.0})
    add_new_rxn(model, "SULabc", "Sulfate transport via ABC system", 0.0, 1000.0, {"so4_e": -1.0, "atp_c": -1.0, "h2o_c": -1.0, "adp_c": 1.0, "pi_c": 1.0, "h_c": 1.0 ,"so4_c": 1.0})

#### EX_h2_e - add to all as it can diffuse to literally everywhere 

In [138]:
def add_h2_mets(model):
    add_new_met(model, "h2_e", "Hydrogen", "H2", 0, "e")
    add_new_met(model, "h2_p", "Hydrogen", "H2", 0, "p")

In [139]:
def add_h2_rxn(model):
    add_new_rxn(model, "EX_h2_e", "Hydrogen exchange", -1000.0, 1000.0, {"h2_e": -1.0})
    add_new_rxn(model, "H2tex", "Hydrogen transport", -1000.0, 1000.0, {"h2_e": -1.0, "h2_p": 1.0})
    add_new_rxn(model, "H2tpp", "Hydrogen transport", -1000.0, 1000.0, {"h2_p": -1.0, "h2_c": 1.0})

In [140]:
def add_h2_all(model):
    if "h2_c" in model.metabolites:
        add_h2_mets(model)
        add_h2_rxn(model)

#### EX_na1_e

In [141]:
def add_na1_mets(model):
    add_new_met(model, "na1_e", "Sodium", "Na", 1, "e")
    add_new_met(model, "na1_c", "Sodium", "Na", 1, "c")

In [142]:
def add_na1_rxn(model):
    add_new_rxn(model, "EX_na1_e", "Sodium exchange", -1000.0, 1000.0, {"na1_e": -1.0})
    add_new_rxn(model, "NAt3_1", "sodium-Proton antiport",-1000, 1000, {"na1_c": -1.0, "h_e":-1.0, "h_c": 1.0, "na1_e": 1.0})

#### EX_fe_2

In [143]:
def add_fe2_mets(model):
    add_new_met(model, "fe2_e", "Iron (Fe2+)", "Fe", 2, "e")


In [144]:
def add_fe2_rxn(model):
    add_new_rxn(model, "EX_fe2_e", "Exchange Reaction for Iron (Fe2+)", -1000, 1000,{"fe2_e": -1.0} )
    add_new_rxn(model, "FE2t", "Iron II transport", 0, 1000,{"fe2_e": -1.0, "fe2_c": 1.0} )

In [145]:
for model_id in no_exr["EX_fe2_e"]:
    model = loaded_model_dict[model_id]
    add_fe2_mets(model)
    add_fe2_rxn(model)


#### Amino acids 

In [146]:
aa_exr = [
    "EX_ala__L_e",
    "EX_arg__L_e",
    "EX_asn__L_e",
    "EX_asp__L_e",
    "EX_cys__L_e",
    "EX_cyst__L_e",
    "EX_gln__L_e",
    "EX_glu__L_e",
    "EX_gly_e",
    "EX_his__L_e",
    "EX_leu__L_e",
    "EX_lys__L_e",
    "EX_mal__L_e",
    "EX_met__L_e",
    "EX_met__D_e",
    "EX_phe__L_e",
    "EX_pro__L_e",
    "EX_ser__D_e",
    "EX_ser__L_e",
    "EX_thr__L_e",
    "EX_trp__L_e",
    "EX_tyr__L_e",
    "EX_tyr__D_e"
]

##### ABC transporters: Ala, Arg, Asn, Asp, Cys, Cystine, Gln, Glu, Gly, His, Ile, Leu, Lys, Met, Val

In [147]:
def add_ala_L_mets(model):
    add_new_met(model, "ala__L_e", "L-Alanine", "C3H7NO2", 0, "e")
def add_ala_L_rxn(model):
    add_new_rxn(model, "EX_ala__L_e", "L-Alanine exchange", -1000.0, 1000.0, {"ala__L_e": -1.0})
    add_new_rxn(model, "ALAabc", "L-Alanine transport via ABC system", 0.0, 1000.0, {"ala__L_e": -1.0, "atp_c": -1.0, "h2o_c": -1.0, "adp_c": 1.0, "pi_c": 1.0, "h_c": 1.0 ,"ala__L_c": 1.0})

In [148]:
def add_arg_L_mets(model):
    add_new_met(model, "arg__L_e", "L-Arginine", "C6H15N4O2", 1, "e")
def add_arg_L_rxn(model):
    add_new_rxn(model, "EX_arg__L_e", "L-Arginine exchange", -1000.0, 1000.0, {"arg__L_e": -1.0})
    add_new_rxn(model, "ARGabc", "L-arginine transport via ABC system", 0.0, 1000.0, {"arg__L_e": -1.0, "atp_c": -1.0, "h2o_c": -1.0, "adp_c": 1.0, "pi_c": 1.0, "h_c": 1.0, "arg__L_c": 1.0})

In [149]:
def add_asn_L_mets(model):
    add_new_met(model, "asn__L_e", "L-Asparagine", "C4H8N2O3", 0, "e")
def add_asn_L_rxn(model):
    add_new_rxn(model, "EX_asn__L_e", "L-Asparagine exchange", -1000.0, 1000.0, {"asn__L_e": -1.0})
    add_new_rxn(model, "ASNabc", "L-Asparagine transport via ABC system", 0.0, 1000.0, {"asn__L_e": -1.0, "atp_c": -1.0, "h2o_c": -1.0, "adp_c": 1.0, "pi_c": 1.0, "h_c": 1.0, "asn__L_c": 1.0})

In [150]:
def add_asp_L_mets(model):
    add_new_met(model, "asp__L_e", "L-Aspartate", "C4H6NO4", -1, "e")
def add_asp_L_rxn(model):
    add_new_rxn(model, "EX_asp__L_e", "L-Aspartate exchange", -1000.0, 1000.0, {"asp__L_e": -1.0})
    add_new_rxn(model, "ASPabc", "L-Aspartate transport via ABC system", 0.0, 1000.0, {"asp__L_e": -1.0, "atp_c": -1.0, "h2o_c": -1.0, "adp_c": 1.0, "pi_c": 1.0, "h_c": 1.0, "asp__L_c": 1.0})

In [151]:
def add_cys_L_mets(model):
    add_new_met(model, "cys__L_e", "L-Cysteine", "C3H7NO2S", 0, "e")
def add_cys_L_rxn(model):
    add_new_rxn(model, "EX_cys__L_e", "L-Cysteine exchange", -1000.0, 1000.0, {"cys__L_e": -1.0})
    add_new_rxn(model, "CYSabc", "L-Cysteine transport via ABC system", 0.0, 1000.0, {"cys__L_e": -1.0, "atp_c": -1.0, "h2o_c": -1.0, "adp_c": 1.0, "pi_c": 1.0, "h_c": 1.0, "cys__L_c": 1.0})

In [152]:
def add_cyst_L_mets(model):
    add_new_met(model, "cyst__L_e", "L-Cystathionine", "C7H14N2O4S", 0, "e")
def add_cyst_L_rxn(model):
    add_new_rxn(model, "EX_cyst__L_e", "L-Cystathionine exchange", -1000.0, 1000.0, {"cyst__L_e": -1.0})
    add_new_rxn(model, "CYSTabc", "L-Cystathionine via ABC system", 0.0, 1000.0, {"cyst__L_e": -1.0, "h2o_c": -1.0, "atp_c": -1.0, "adp_c":1.0, "pi_c":1.0, "h_c": 1.0, "cyst__L_c": 1.0})

In [153]:
def add_gln_L_mets(model):
    add_new_met(model, "gln__L_e", "L-Glutamine", "C5H10N2O3", 0, "e")
def add_gln_L_rxn(model):
    add_new_rxn(model, "EX_gln__L_e", "L-Glutamine exchange", -1000.0, 1000.0, {"gln__L_e": -1.0})
    add_new_rxn(model, "GLNabc", "Glutamine transport via ABC system", 0.0, 1000.0, {"gln__L_e": -1.0, "h2o_c": -1.0, "atp_c": -1.0, "adp_c":1.0, "pi_c":1.0, "h_c": 1.0, "gln__L_c": 1.0})

In [154]:
def add_glu__L_mets(model):
    add_new_met(model, "glu__L_e", "L-Glutamate", "C5H8NO4", -1, "e")
def add_glu__L_rxn(model):
    add_new_rxn(model, "EX_glu__L_e", "L-Glutamate exchange", -1000.0, 1000.0, {"glu__L_e": -1.0})
    add_new_rxn(model, "GLUabc", "Glutamate transport via ABC system", 0.0, 1000.0, {"glu__L_e": -1.0, "h2o_c": -1.0, "atp_c": -1.0, "adp_c":1.0, "pi_c":1.0, "h_c": 1.0, "glu__L_c": 1.0})

In [155]:
def add_gly_mets(model):
    add_new_met(model, "gly_e", "Glycine", "C2H5NO2", 0, "e")
def add_gly_rxn(model):
    add_new_rxn(model, "EX_gly_e", "Glycine exchange", -1000.0, 1000.0, {"gly_e": -1.0})
    add_new_rxn(model, "GLYabc", "Glycine transport via ABC system", 0.0, 1000.0, {"gly_e": -1.0, "h2o_c": -1.0, "atp_c": -1.0, "adp_c":1.0, "pi_c":1.0, "h_c": 1.0, "gly_c": 1.0})

In [156]:
def add_his__L_mets(model):
    add_new_met(model, "his__L_e", "L-Histidine", "C6H9N3O2", 0, "e")
def add_his__L_rxn(model):
    add_new_rxn(model, "EX_his__L_e", "L-Histidine exchange", -1000.0, 1000.0, {"his__L_e": -1.0})
    add_new_rxn(model, "HISabc", "L_Histidine transport via ABC system", 0.0, 1000.0, {"his__L_e": -1.0, "h2o_c": -1.0, "atp_c": -1.0, "adp_c":1.0, "pi_c":1.0, "h_c": 1.0, "his__L_c": 1.0})

In [157]:
def add_lys__L_mets(model):
    add_new_met(model, "lys__L_e", "L-Lysine", "C6H15N2O2", 1, "e")
def add_lys__L_rxn(model):
    add_new_rxn(model, "EX_lys__L_e", "L-Lysine exchange", -1000.0, 1000.0, {"lys__L_e": -1.0})
    add_new_rxn(model, "LYSabc", "L-Lysine transport via ABC system", 0.0, 1000.0, {"lys__L_e": -1.0, "h2o_c": -1.0, "atp_c": -1.0, "adp_c":1.0, "pi_c":1.0, "h_c": 1.0, "lys__L_c": 1.0})

In [158]:
def add_leu__L_mets(model):
    add_new_met(model, "leu__L_e", "L-Leucine", "C6H13NO2", 0, "e")
def add_leu__L_rxn(model):
    add_new_rxn(model, "EX_leu__L_e", "L-Leucine exchange", -1000.0, 1000.0, {"leu__L_e": -1.0})
    add_new_rxn(model, "LEUabc", "L-Leucine transport via ABC system", 0.0, 1000.0, {"leu__L_e": -1.0, "h2o_c": -1.0, "atp_c": -1.0, "adp_c":1.0, "pi_c":1.0, "h_c": 1.0, "leu__L_c": 1.0})

In [159]:
def add_met__L_mets(model):
    add_new_met(model, "met__L_e", "L-Methionine", "C5H11NO2S", 0, "e")
def add_met__L_rxn(model):
    add_new_rxn(model, "EX_met__L_e", "L-Methionine exchange", -1000.0, 1000.0, {"met__L_e": -1.0})
    add_new_rxn(model, "METabc", "L-Methionine via ABC system", 0.0, 1000.0, {"met__L_e": -1.0, "h2o_c": -1.0, "atp_c": -1.0, "adp_c":1.0, "pi_c":1.0, "h_c": 1.0, "met__L_c": 1.0})

In [160]:
def add_met__D_mets(model):
    add_new_met(model, "met__D_e", "D-Methionine", "C5H11NO2S", 0, "e")
def add_met__D_rxn(model):
    add_new_rxn(model, "EX_met__D_e", "D-Methionine exchange", -1000.0, 1000.0, {"met__D_e": -1.0})
    add_new_rxn(model, "METDabc", "D-Methionine via ABC system", 0.0, 1000.0, {"met__D_e": -1.0, "h2o_c": -1.0, "atp_c": -1.0, "adp_c":1.0, "pi_c":1.0, "h_c": 1.0, "met__D_c": 1.0})

In [161]:
def add_phe_L_mets(model):
    add_new_met(model, "phe__L_e", "L-Phenylalanine", "C9H11NO2", 0, "e")
def add_phe_L_rxn(model):
    add_new_rxn(model, "EX_phe__L_e", "L-Phenylalanine exchange", -1000.0, 1000.0, {"phe__L_e": -1.0})
    add_new_rxn(model, "PHEabc", "L-Phenylalanine ABC system", 0.0, 1000.0, {"phe__L_e": -1.0, "atp_c": -1.0, "h2o_c": -1.0, "adp_c": 1.0, "pi_c": 1.0, "h_c": 1.0, "phe__L_c": 1.0})

In [162]:
def add_trp_L_mets(model):
    add_new_met(model, "trp__L_e", "L-Tryptophan", "C11H12N2O2", 0, "e")
def add_trp_L_rxn(model):
    add_new_rxn(model, "EX_trp__L_e", "EX_trp__L_e", -1000.0, 1000.0, {"trp__L_e": -1.0})
    add_new_rxn(model, "TRPabc", "TRP transport ABC system", 0.0, 1000.0, {"trp__L_e": -1.0, "atp_c": -1.0, "h2o_c": -1.0, "adp_c": 1.0, "pi_c": 1.0, "h_c": 1.0, "trp__L_c": 1.0})

In [163]:
def add_tyr__L_mets(model):
    add_new_met(model, "tyr__L_e", "L-Tyrosine", "C9H11NO3", 0, "e")
def add_tyr__L_rxn(model):
    add_new_rxn(model, "EX_tyr__L_e", "L-Tyrosine exchange", -1000.0, 1000.0, {"tyr__L_e": -1.0})
    add_new_rxn(model, "TYRabc", "L-Tyrosine transport via ABC transport", 0.0, 1000.0, {"tyr__L_e": -1.0, "atp_c": -1.0, "h2o_c": -1.0, "adp_c": 1.0, "pi_c": 1.0,  "h_c": 1.0, "tyr__L_c": 1.0})

In [164]:
def add_tyr__D_mets(model):
    add_new_met(model, "tyr__D_e", "D-Tyrosine", "C9H11NO3", 0, "e")
def add_tyr__D_rxn(model):
    add_new_rxn(model, "EX_tyr__D_e", "D-Tyrosine exchange", -1000.0, 1000.0, {"tyr__D_e": -1.0})
    add_new_rxn(model, "TYRabc_D", "D-Tyrosine transport via ABC transport", 0.0, 1000.0, {"tyr__D_e": -1.0, "atp_c": -1.0, "h2o_c": -1.0, "adp_c": 1.0, "pi_c": 1.0,  "h_c": 1.0, "tyr__D_c": 1.0})

##### Sodium Symport: Pro, Thr

In [165]:
def add_pro_L_mets(model):
    add_new_met(model, "pro__L_e", "L-Proline", "C5H9NO2", 0, "e")
def add_pro_L_rxn(model):
    add_new_rxn(model, "EX_pro__L_e", "L-Proline exchange", -1000.0, 1000.0, {"pro__L_e": -1.0})
    add_new_rxn(model, "PROt4", "L-Proline Sodium symport", 0.0, 1000.0, {"pro__L_e": -1.0, "na1_e": -1.0, "na1_c": 1.0, "pro__L_c": 1.0})

In [166]:
def add_thr__L_mets(model):
    add_new_met(model, "thr__L_e", "L-Threonine", "C4H9NO3", 0, "e")
def add_thr__L_rxn(model):
    add_new_rxn(model, "EX_thr__L_e", "L-Threonine exchange", -1000.0, 1000.0, {"thr__L_e": -1.0})
    add_new_rxn(model, "THRt4", "L-Threonine transport via sodium symport", 0.0, 1000.0, {"thr__L_e": -1.0, "na1_e": -1.0, "na1_c": 1.0,  "thr__L_c": 1.0})

##### Proton Symport: L-Malate, Serine

In [167]:
def add_mal__L_mets(model):
    add_new_met(model, "mal__L_e", "L-Malate", "C4H4O5", 0, "e")
def add_mal__L_rxn(model):
    add_new_rxn(model, "EX_mal__L_e", "L-Malate exchange", -1000.0, 1000.0, {"mal__L_e": -1.0})
    add_new_rxn(model, "MALt2r", "L Malate transport via proton symport", -1000.0, 1000.0, {"mal__L_e": -1.0, "h_e": -1.0, "h_c": 1.0, "mal__L_c": 1.0}) 

In [168]:
def add_ser__D_mets(model):
    add_new_met(model, "ser__D_e", "D-Serine", "C3H7NO3", 0, "e")
def add_ser__D_rxn(model):
    add_new_rxn(model, "EX_ser__D_e", "D-Serine exchange", -1000.0, 1000.0, {"ser__D_e": -1.0})
    add_new_rxn(model, "DSERt2", "D-serine transport via proton symport", 0.0, 1000.0, {"ser__D_e": -1.0, "h_e": -1.0, "h_c": 1.0, "ser__D_c": 1.0})

In [169]:
def add_ser__L_mets(model):
    add_new_met(model, "ser__L_e", "L-Serine", "C3H7NO3", 0, "e")
def add_ser__L_rxn(model):
    add_new_rxn(model, "EX_ser__L_e", "L-Serine exchange", -1000.0, 1000.0, {"ser__L_e": -1.0})
    add_new_rxn(model, "SERt2r", "L-serine transport via proton symport", 0.0, 1000.0, {"ser__L_e": -1.0, "h_e": -1.0, "h_c": 1.0, "ser__L_c": 1.0})

In [170]:
AMINO_ACID_MAP = {
    "ala__L": {"mets": add_ala_L_mets, "rxns": add_ala_L_rxn},
    "arg__L": {"mets": add_arg_L_mets, "rxns": add_arg_L_rxn},
    "asn__L": {"mets": add_asn_L_mets, "rxns": add_asn_L_rxn},
    "asp__L": {"mets": add_asp_L_mets, "rxns": add_asp_L_rxn},
    "cys__L": {"mets": add_cys_L_mets, "rxns": add_cys_L_rxn},
    "cyst__L": {"mets": add_cyst_L_mets, "rxns": add_cyst_L_rxn},
    "gln__L": {"mets": add_gln_L_mets, "rxns": add_gln_L_rxn},
    "glu__L": {"mets": add_glu__L_mets, "rxns": add_glu__L_rxn},
    "gly": {"mets": add_gly_mets, "rxns": add_gly_rxn},
    "his__L": {"mets": add_his__L_mets, "rxns": add_his__L_rxn},
    "leu__L": {"mets": add_leu__L_mets, "rxns": add_leu__L_rxn},
    "lys__L": {"mets": add_lys__L_mets, "rxns": add_lys__L_rxn},
    "mal__L": {"mets": add_mal__L_mets, "rxns": add_mal__L_rxn},
    "met__L": {"mets": add_met__L_mets, "rxns": add_met__L_rxn},
    "met__D": {"mets": add_met__D_mets, "rxns": add_met__D_rxn},
    "phe__L": {"mets": add_phe_L_mets, "rxns": add_phe_L_rxn},
    "pro__L": {"mets": add_pro_L_mets, "rxns": add_pro_L_rxn},
    "ser__D": {"mets": add_ser__D_mets, "rxns": add_ser__D_rxn},
    "ser__L": {"mets": add_ser__L_mets, "rxns": add_ser__L_rxn},
    "thr__L": {"mets": add_thr__L_mets, "rxns": add_thr__L_rxn},
    "trp__L": {"mets": add_trp_L_mets, "rxns": add_trp_L_rxn},
    "tyr__L": {"mets": add_tyr__L_mets, "rxns": add_tyr__L_rxn},
    "tyr__D": {"mets": add_tyr__D_mets, "rxns": add_tyr__D_rxn},

}

## Main 

In [177]:
# remove as should not be in all 
t_r = ["FE3PYOVDL2", "EX_fe3pyovd_kt_e", "EX_pyovd_kt_e"]

In [178]:
for model_id, model in loaded_model_dict.items():
    print(model_id)
    model_nr = model_id.split("_")[1]
    add_non_aa(model_id, model)
    add_glc_all(model)
    add_h2_all(model)
    add_fru_mets(model)
    add_fru_rxn(model)
    for ex_r in no_exr.keys():
        if ex_r in aa_exr:
            add_amino_acids(model, ex_r)
    for r in t_r:
        if r in model.reactions:
            model.reactions.get_by_id(r).bounds = (0, 0)
    write_sbml_model(model, os.path.join(model_dir_save, f"{model_nr}_ex.xml"))

m_504
m_796
m_364
m_1018
m_1252
m_778
m_1234
m_1114
m_428
m_892
m_939
m_790
m_1124
m_895
m_230
m_1432
m_163
m_1056
m_161
m_1101
m_1338
m_1208
m_946
m_2862
m_397
m_352
m_459
m_262
m_638
m_100
m_997
m_1334
m_2751
m_2774
m_709
m_1391
m_2872
m_868
m_1357
m_761
m_1080
m_1350
m_947
m_1362
m_793
m_1167
m_644
m_978
m_867
m_1174


In [ ]:
for modid, model in loaded_model_dict.items():
    
    for r in t_r:
        if r in model.reactions:
            sol = model.slim_optimize()
            print(f"Flux before knocking out {r}: {sol}")
            model.reactions.get_by_id(r).bounds = (0, 0)
            output_path = os.path.join(model_dir, f"{model_id}_or_mb1_mdr_rdr_dp_lib_bz.xml")
            write_sbml_model(model, output_path) 
            
    sol2 = model.slim_optimize()
    print(f"Final cumulative knockout optimization: {sol2} for {modid}")

Flux before knocking out FE3PYOVDL2: 16.96678217934754
Flux before knocking out EX_fe3pyovd_kt_e: 16.96678217934754
Flux before knocking out EX_pyovd_kt_e: 16.96678217934754
Final cumulative knockout optimization: 16.96678217934754 for m_504
Flux before knocking out FE3PYOVDL2: 15.608400100671048
Flux before knocking out EX_fe3pyovd_kt_e: 15.608400100671048
Flux before knocking out EX_pyovd_kt_e: 15.608400100671048
Final cumulative knockout optimization: 15.608400100671048 for m_796
Flux before knocking out FE3PYOVDL2: 17.056012798832004
Flux before knocking out EX_fe3pyovd_kt_e: 17.056012798832004
Flux before knocking out EX_pyovd_kt_e: 17.056012798832004
Final cumulative knockout optimization: 17.056012798832004 for m_364
Flux before knocking out FE3PYOVDL2: 17.146932821815177
Flux before knocking out EX_fe3pyovd_kt_e: 17.146932821815177
Flux before knocking out EX_pyovd_kt_e: 17.146932821815177
Final cumulative knockout optimization: 17.146932821815177 for m_1018
Flux before knockin